<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/06a_langfuse_custom_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6a: Wiring RAGAS, DeepEval, and Promptfoo Scores into Langfuse as Custom Scores

**Goal:** Wire scores from Phases 2 through 5c into Langfuse as structured custom scores attached to traced runs, building the production observability layer described in the Talabat prep's "drift monitoring = Langfuse Phase 6" pillar.

**Design decision, built to be flip-ready correctly, not just superficially:** this notebook does not generate its own scores, it wires in scores already produced upstream. Flipping `SIMULATED_OUTPUT` to `False` here is therefore not sufficient by itself to make this notebook "real": each upstream phase's saved results file carries its own `"simulated"` field, and this notebook checks that field per source before pushing anything to a live Langfuse dashboard as genuine. A score from a still-simulated upstream phase is pushed to Langfuse tagged as simulated, never silently presented as real just because this notebook's own flag was flipped.

**Tools:** Langfuse v4, reads saved JSON output from Phases 2a/2b, 3a/3b, 4a/4b, 5a/5b/5c

**Date:** July 2026

**Status:** In progress. Wiring logic is written to run for real the moment Langfuse credentials exist and is not blocked on Gemini or Claude billing at all, since this phase moves already-computed numbers, it makes no model calls of its own.

In [2]:
# Cell 2: Mount Drive and load all upstream results
# Corrected filenames, checked against each notebook's actual save cell
# rather than guessed.

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

UPSTREAM_SOURCES = {
    "phase02a": "phase02a_gemini_judge_results.json",
    "phase02b": "phase02b_claude_judge_results.json",
    "phase03a": "phase03a_deepeval_rag_results.json",
    "phase03b": "phase03b_governance_metrics_results.json",
    "phase04a": "phase04a_aspect_critic_results.json",
    "phase04b": "phase04b_judge_alignment_results.json",
    "phase05a": "phase05a_promptfoo_owasp_results.json",
    "phase05b": "phase05b_promptfoo_owasp_agentic_results.json",
    "phase05c": "phase05c_mitre_atlas_mapping_results.json",
}

upstream_data = {}
upstream_status = {}

for phase_key, filename in UPSTREAM_SOURCES.items():
    path = DRIVE_PATH + filename
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        upstream_data[phase_key] = data
        is_simulated = data.get("simulated", None)
        upstream_status[phase_key] = is_simulated
    else:
        upstream_data[phase_key] = None
        upstream_status[phase_key] = "MISSING"

print("UPSTREAM SOURCE STATUS")
print("=" * 60)
for phase_key, status in upstream_status.items():
    if status == "MISSING":
        flag = "MISSING FILE"
    elif status is True:
        flag = "SIMULATED"
    elif status is False:
        flag = "REAL"
    else:
        flag = "NO 'simulated' FIELD, mixed real/simulated by design (05c only)"
    print(f"  {phase_key}: {flag}")

real_count = sum(1 for v in upstream_status.values() if v is False)
simulated_count = sum(1 for v in upstream_status.values() if v is True)
print()
print(f"Real: {real_count} | Simulated: {simulated_count} | "
      f"Missing/mixed: {len(upstream_status) - real_count - simulated_count}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
UPSTREAM SOURCE STATUS
  phase02a: SIMULATED
  phase02b: SIMULATED
  phase03a: SIMULATED
  phase03b: SIMULATED
  phase04a: SIMULATED
  phase04b: SIMULATED
  phase05a: SIMULATED
  phase05b: SIMULATED
  phase05c: NO 'simulated' FIELD, mixed real/simulated by design (05c only)

Real: 0 | Simulated: 8 | Missing/mixed: 1


In [3]:
# Cell 3: Install packages

!pip install langfuse pandas --quiet

print("Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
